# Essential Imports and Configurations

In [20]:
import pandas as pd
import sqlalchemy as sqla
import os
import re

from pathlib import Path
from sqlalchemy import create_engine, text

from dotenv import load_dotenv
from typing import Dict, Optional

In [21]:
load_dotenv()

user: str = os.getenv("DB_USER")
password: str = os.getenv("DB_PASSWORD")
host: str = os.getenv("DB_HOST")
port: str = os.getenv("PORT")
database: str = os.getenv("DB_NAME")

In [22]:
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', 1000)         # Prevents wrapping

In [23]:
def get_connection() -> sqla.Engine:
    engine: sqla.Engine = create_engine(
        f"postgresql://{user}:{password}@{host}:{port}/{database}"
    )
    return engine

try:
    engine: sqla.Engine = get_connection()
    print(f"Connection to the {host} for user {user} created successfully.")

except Exception as ex:
    print("Connection could not be made due to the following error:\n", ex)

Connection to the localhost for user postgres created successfully.


In [24]:
def ingest_data() -> None:
    """
    Populates the Cyber Databoard database with the CSV files data.
    """
    engine: sqla.Engine = get_connection()
    parent: str = (Path.cwd())
    path: Path = Path(f"{parent}/data/")
    day_regex: str = r"[\w\-.]+(?=-[Ww]orkingHours)"
    type_regex: str = r"(?<=-[Ww]orkingHours-)[\w\-.]+(?=.pcap)"

    for file_name in path.iterdir():
        with open(file_name, mode='r') as file:
            datafile: pd.DataFrame = pd.read_csv(file)
            day_match: re.Match = re.search(day_regex, str(file_name.name))
            type_match: re.Match = re.search(type_regex, str(file_name))
            if day_match:
                day_match: str = day_match.group().replace("_", " ")
            if type_match:
                type_match: str = type_match.group().replace("_", " ")
            else:
                type_match: str = "Working_Hours"

            table_name: str = f"{day_match}_{type_match}".replace(" ", "_")
            
            datafile.to_sql(
                f"{table_name}",
                engine, 
                if_exists="replace",
                index=False
            )


# Data Analysis

In [25]:
def query_database(query: str, para_bindings: Optional[Dict[str, str | int]] = None) -> pd.DataFrame:
    with engine.connect() as connection:
        query: sqla.Text = text(query)

        result: sqla.CursorResult = connection.execute(query, para_bindings)
        df: pd.DataFrame = pd.DataFrame(result)
        
        return df

# Friday_DDOS Analysis

This dataset seems to contain the data from DDOS attacks that occured during Friday Afternoon.

In [26]:
Friday_DDOS: pd.DataFrame = query_database('SELECT * FROM "Friday_Afternoon_DDos"')

In [27]:
Friday_DDOS.info()

<class 'pandas.DataFrame'>
RangeIndex: 225745 entries, 0 to 225744
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Flow ID                       225745 non-null  str    
 1    Source IP                    225745 non-null  str    
 2    Source Port                  225745 non-null  int64  
 3    Destination IP               225745 non-null  str    
 4    Destination Port             225745 non-null  int64  
 5    Protocol                     225745 non-null  int64  
 6    Timestamp                    225745 non-null  str    
 7    Flow Duration                225745 non-null  int64  
 8    Total Fwd Packets            225745 non-null  int64  
 9    Total Backward Packets       225745 non-null  int64  
 10  Total Length of Fwd Packets   225745 non-null  int64  
 11   Total Length of Bwd Packets  225745 non-null  int64  
 12   Fwd Packet Length Max        225745 non-null  int64  


In [28]:
Friday_DDOS.describe()

C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\emera\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,225745.000000,225745.00000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,2.257410e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,225745.000000,225745.0,225745.0,225745.0,225745.000000,225745.000000,2.257450e+05,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.0,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.000000,225745.0,225745.0,225745.0,225745.0,225745.0,225745.0,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05
mean,38257.568402,8879.61946,7.600288,1.624165e+07,4.874916,4.572775,939.463346,5.960477e+03,538.535693,27.882221,164.826715,214.907242,2735.585147,16.718776,890.536849,1230.172938,inf,inf,1.580587e+06,4.248569e+06,1.348977e+07,2.811855e+04,1.539652e+07,2.540610e+06,5.195207e+06,1.299434e+07,2.073698e+05,6.564701e+06,9.476322e+05,1.610306e+06,4.567514e+06,2.257817e+05,0.033223,0.0,0.0,0.0,111.522718,106.789023,1.261508e+04,1.641693e+03,8.072595,3226.045339,515.002137,1085.593207,2.789906e+06,0.002671,0.033223,0.000120,0.351162,0.504463,0.140752,0.0,0.000120,1.005821,574.568843,164.826715,890.536849,111.522718,0.0,0.0,0.0,0.0,0.0,0.0,4.874916,939.463346,4.572775,5.960477e+03,4247.436922,601.048635,3.311497,21.482753,1.848261e+05,1.293436e+04,2.080849e+05,1.776201e+05,1.032214e+07,3.611943e+06,1.287813e+07,7.755355e+06
std,23057.302075,19754.64740,3.881586,3.152437e+07,15.422874,21.755356,3249.403484,3.921834e+04,1864.128991,163.324159,504.892965,797.411073,3705.123460,50.480568,1120.324921,1733.201267,NaN,NaN,2.701596e+06,7.622819e+06,2.670172e+07,7.598100e+05,3.160826e+07,5.934694e+06,1.078635e+07,2.748870e+07,3.795228e+06,2.198455e+07,4.586374e+06,5.475778e+06,1.617865e+07,4.019290e+06,0.179220,0.0,0.0,0.0,375.790727,511.765795,1.106701e+05,1.989593e+04,15.767713,3813.134850,559.064495,1269.558714,4.115941e+06,0.051614,0.179220,0.010936,0.477334,0.499981,0.347766,0.0,0.010936,1.430781,626.096202,504.892965,1120.324921,375.790727,0.0,0.0,0.0,0.0,0.0,0.0,15.422874,3249.403484,21.755356,3.921834e+04,8037.781019,4319.720339,12.270018,4.166799,7.979250e+05,2.102737e+05,9.002350e+05,7.842602e+05,2.185303e+0

In [29]:
Friday_DDOS.isnull().any()

Flow ID                         False
 Source IP                      False
 Source Port                    False
 Destination IP                 False
 Destination Port               False
 Protocol                       False
 Timestamp                      False
 Flow Duration                  False
 Total Fwd Packets              False
 Total Backward Packets         False
Total Length of Fwd Packets     False
 Total Length of Bwd Packets    False
 Fwd Packet Length Max          False
 Fwd Packet Length Min          False
 Fwd Packet Length Mean         False
 Fwd Packet Length Std          False
Bwd Packet Length Max           False
 Bwd Packet Length Min          False
 Bwd Packet Length Mean         False
 Bwd Packet Length Std          False
Flow Bytes/s                     True
 Flow Packets/s                 False
 Flow IAT Mean                  False
 Flow IAT Std                   False
 Flow IAT Max                   False
 Flow IAT Min                   False
Fwd IAT Tota

In [30]:
Friday_DDOS['Flow Bytes/s'].isnull().sum()

# Since there are so little missing values, we can just drop them

print(f"Before removing missing values: {Friday_DDOS.shape}")
Friday_DDOS.dropna(inplace=True)
print(f"After removing missing values: {Friday_DDOS.shape}")

Before removing missing values: (225745, 85)
After removing missing values: (225741, 85)


In [31]:
ports = query_database('SELECT " Destination Port", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Destination Port" ORDER BY COUNT(*) DESC')

In [32]:
ports.head(15)

,Destination Port,total
0,80,136951
1,53,31950
2,443,13485
3,8080,510
4,123,362
5,22,342
6,137,274
7,389,261
8,88,173
9,21,167


In [33]:
packet_size = query_database('SELECT " Average Packet Size", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Average Packet Size" ORDER BY total DESC')

In [34]:
packet_size.head(15)

,Average Packet Size,total
0,7.500000,23069
1,7.200000,22412
2,1454.125000,15798
3,1292.555556,10349
4,7.000000,9741
5,1661.000000,9358
6,1291.888889,9131
7,1453.375000,8848
8,9.000000,7871
9,1661.857143,6634


In [35]:
protocols = query_database('SELECT " Protocol", COUNT(*) AS total FROM "Friday_Afternoon_DDos" GROUP BY  " Protocol" ORDER BY total DESC')

Protocol 6 = TCP \
Protocol 17 = UDP \
Protocol 0 = Nan  

In [36]:
protocols.head()

,Protocol,total
0,6,192820
1,17,32871
2,0,54
